In [12]:
pip install pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: C:\Users\ksili\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [8]:
pip install seaborn

SyntaxError: invalid syntax (3435186165.py, line 1)

In [ ]:
pip install glob

In [ ]:
pip install zipfile

In [21]:
import glob, zipfile
import pandas as pd
import pickle
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, r2_score

In [16]:
pd.set_option("mode.string_storage", "python") 
COLS = ["FlightDate", "Reporting_Airline", "Origin", "Dest", "CRSDepTime",
        "CRSElapsedTime", "Distance", "DepDelay", "ArrDelay", "ArrDel15",
        "Cancelled", "Diverted"]


def read(path):
    with zipfile.ZipFile(path) as z:
        name = [n for n in z.namelist() if n.endswith(".csv")][0]
        return pd.read_csv(z.open(name), usecols=COLS)


df = pd.concat([read(f) for f in sorted(glob.glob("us-flights-2026-raw/monthly-zips/*.zip"))],
               ignore_index=True)

df = df[(df.Cancelled == 0) & (df.Diverted == 0)].drop(columns=["Cancelled", "Diverted"])
df = df.dropna(subset=["ArrDelay", "DepDelay", "CRSElapsedTime"])

df["FlightDate"] = pd.to_datetime(df.FlightDate)
df["Month"] = df.FlightDate.dt.month
df["DayOfWeek"] = df.FlightDate.dt.dayofweek
df["DepHour"] = df.CRSDepTime // 100 % 24
df["ArrDel15"] = df.ArrDel15.fillna(df.ArrDelay >= 15).astype(int)

df.to_pickle("flights_2026.pkl")  
print(df.shape)
print(df.head())

(3402619, 13)
  FlightDate Reporting_Airline Origin Dest  CRSDepTime  DepDelay  ArrDelay  \
0 2026-01-01                AA    JFK  LAX         700      -3.0      43.0   
1 2026-01-02                AA    JFK  LAX         700      -6.0       1.0   
2 2026-01-03                AA    JFK  LAX         700      -2.0     -10.0   
3 2026-01-04                AA    JFK  LAX         729      -4.0     -18.0   
4 2026-01-05                AA    JFK  LAX         729      -9.0      -3.0   

   ArrDel15  CRSElapsedTime  Distance  Month  DayOfWeek  DepHour  
0         1           381.0    2475.0      1          3        7  
1         0           381.0    2475.0      1          4        7  
2         0           381.0    2475.0      1          5        7  
3         0           387.0    2475.0      1          6        7  
4         0           387.0    2475.0      1          0        7  


In [19]:
F = ["DepDelay", "Month", "DayOfWeek", "DepHour", "Distance", "CRSElapsedTime",
     "Reporting_Airline", "Origin", "Dest"]

df = pd.read_pickle("flights_2026.pkl")
for c in ["Reporting_Airline", "Origin", "Dest"]:
    df[c] = df[c].astype("category").cat.codes

train = df[df.FlightDate < "2026-06-01"]
test = df[df.FlightDate >= "2026-06-01"]

base = DummyRegressor().fit(train[F], train.ArrDelay)
m = HistGradientBoostingRegressor(max_iter=300, random_state=0).fit(train[F], train.ArrDelay)

for name, p in [("baseline", base.predict(test[F])), ("модель", m.predict(test[F]))]:
    print(f"{name:9} MAE {mean_absolute_error(test.ArrDelay, p):5.1f} мин   "
          f"R2 {r2_score(test.ArrDelay, p):6.3f}")

baseline  MAE  30.2 мин   R2 -0.006
модель    MAE  10.9 мин   R2  0.862
